# Student Alcohol Consumption Analysis

**Exploring the relationship between student demographics, academics, and alcohol consumption patterns.**

## Project Overview

This project analyzes a student survey dataset to understand factors associated with alcohol consumption among secondary school students. We examine demographics, family background, study habits, and social factors to identify patterns in weekday and weekend drinking.

## Learning Objectives

1. Analyze survey-based social science data
2. Work with ordinal variables (alcohol consumption levels 1-5)
3. Explore relationships between multiple categorical and numerical features
4. Apply appropriate statistical tests for ordinal data
5. Create clear visualizations of social science findings

## Business / Research Problem

Understanding student alcohol consumption patterns helps:
- **Schools** design targeted intervention programs
- **Public health** develop prevention strategies
- **Researchers** identify at-risk demographics

**Key question:** *What student characteristics are most strongly associated with higher alcohol consumption?*

## Why This Analysis Matters

Underage drinking is a significant public health concern affecting academic performance, health, and social development.

## Dataset Overview

Key features include: school, sex, age, family size, parental status, parental education, travel time, study time, failures, support, activities, internet access, romantic relationship, family relations, free time, going out, weekday alcohol (Dalc), weekend alcohol (Walc), health, absences, and grades (G1, G2, G3).

**Alcohol levels:** 1 (very low) to 5 (very high)

## Dataset Source and License Notes- **Source:** [Kaggle - Student Alcohol Consumption](https://www.kaggle.com/datasets/uciml/student-alcohol-consumption)- **Original:** UCI ML Repository (Cortez and Silva, 2008)- **License:** CC BY 4.0

## Environment Setup

In [1]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded.')

All imports loaded.


## Configuration / Constants

In [3]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'uciml/student-alcohol-consumption'
SIGNIFICANCE_LEVEL = 0.05
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'Available files: {csv_files}')

# Load math student data (usually student-mat.csv)
df = pd.read_csv(os.path.join(path, csv_files[0]))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

  0%|          | 0.00/18.4k [00:00<?, ?B/s]

100%|██████████| 18.4k/18.4k [00:00<00:00, 400kB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\uciml\student-alcohol-consumption\versions\2
Available files: ['student-mat.csv', 'student-por.csv']
Loaded 395 rows and 33 columns


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## Data Validation Checks

In [5]:
print(f'Shape: {df.shape}')
print(f'Missing: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nAlcohol columns:')
for col in ['Dalc', 'Walc']:
    if col in df.columns:
        print(f'  {col}: {df[col].value_counts().sort_index().to_dict()}')
df.describe()

Shape: (395, 33)
Missing: 0
Duplicates: 0

Alcohol columns:
  Dalc: {1: 276, 2: 75, 3: 26, 4: 9, 5: 9}
  Walc: {1: 151, 2: 85, 3: 80, 4: 51, 5: 28}


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000


## Data Cleaning

In [6]:
df = df.drop_duplicates()

# Create total alcohol consumption
if 'Dalc' in df.columns and 'Walc' in df.columns:
    df['total_alc'] = df['Dalc'] + df['Walc']
    df['high_drinker'] = (df['total_alc'] >= 6).astype(int)
    print(f'High drinkers (Dalc+Walc >= 6): {df["high_drinker"].sum()} ({df["high_drinker"].mean()*100:.1f}%)')

print(f'Rows: {len(df)}')

High drinkers (Dalc+Walc >= 6): 74 (18.7%)
Rows: 395


## Exploratory Data Analysis

In [7]:
print('=== Demographics ===')
if 'sex' in df.columns:
    print(f'Gender: {df["sex"].value_counts().to_dict()}')
if 'age' in df.columns:
    print(f'Age range: {df["age"].min()}-{df["age"].max()}, mean={df["age"].mean():.1f}')

print(f'\n=== Grade Stats ===')
for gcol in ['G1', 'G2', 'G3']:
    if gcol in df.columns:
        print(f'{gcol}: mean={df[gcol].mean():.2f}, median={df[gcol].median():.1f}')

=== Demographics ===
Gender: {'F': 208, 'M': 187}
Age range: 15-22, mean=16.7

=== Grade Stats ===
G1: mean=10.91, median=11.0
G2: mean=10.71, median=11.0
G3: mean=10.42, median=11.0


## Univariate Analysis

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

if 'Dalc' in df.columns:
    df['Dalc'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue')
    axes[0,0].set_title('Weekday Alcohol (Dalc)')

if 'Walc' in df.columns:
    df['Walc'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='coral')
    axes[0,1].set_title('Weekend Alcohol (Walc)')

if 'age' in df.columns:
    axes[0,2].hist(df['age'], bins=10, color='mediumseagreen', edgecolor='white')
    axes[0,2].set_title('Age Distribution')

if 'G3' in df.columns:
    axes[1,0].hist(df['G3'], bins=20, color='mediumpurple', edgecolor='white')
    axes[1,0].set_title('Final Grade (G3) Distribution')

if 'absences' in df.columns:
    axes[1,1].hist(df['absences'], bins=20, color='orange', edgecolor='white')
    axes[1,1].set_title('Absences Distribution')

if 'total_alc' in df.columns:
    df['total_alc'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], color='brown')
    axes[1,2].set_title('Total Alcohol (Dalc + Walc)')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if 'Walc' in df.columns and 'G3' in df.columns:
    sns.boxplot(data=df, x='Walc', y='G3', ax=axes[0], palette='viridis')
    axes[0].set_title('Final Grade by Weekend Alcohol')

if 'Walc' in df.columns and 'sex' in df.columns:
    pd.crosstab(df['sex'], df['Walc'], normalize='index').plot(kind='bar', ax=axes[1], stacked=True)
    axes[1].set_title('Weekend Alcohol by Gender')
    axes[1].legend(title='Walc')

if 'Walc' in df.columns and 'absences' in df.columns:
    sns.boxplot(data=df, x='Walc', y='absences', ax=axes[2], palette='Set2')
    axes[2].set_title('Absences by Weekend Alcohol')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [10]:
# Alcohol consumption by key factors
if 'Walc' in df.columns:
    print('=== Mean Weekend Alcohol by Factor ===')
    for factor in ['sex', 'romantic', 'internet', 'higher', 'activities']:
        if factor in df.columns:
            print(f'\n{factor}:')
            print(df.groupby(factor)['Walc'].mean().round(3))

=== Mean Weekend Alcohol by Factor ===

sex:
sex
F    1.957
M    2.663
Name: Walc, dtype: float64

romantic:
romantic
no     2.300
yes    2.273
Name: Walc, dtype: float64

internet:
internet
no     2.258
yes    2.298
Name: Walc, dtype: float64

higher:
higher
no     2.850
yes    2.261
Name: Walc, dtype: float64

activities:
activities
no     2.340
yes    2.244
Name: Walc, dtype: float64


## Statistical Checks / Hypothesis Testing

In [11]:
# Do males drink more than females?
if 'sex' in df.columns and 'Walc' in df.columns:
    male_alc = df[df['sex'] == 'M']['Walc']
    female_alc = df[df['sex'] == 'F']['Walc']
    u_stat, p_val = stats.mannwhitneyu(male_alc, female_alc, alternative='two-sided')
    print(f'Mann-Whitney U (Male vs Female Walc): U={u_stat:.0f}, p={p_val:.4f}')
    print(f'Result: {"Significant" if p_val < SIGNIFICANCE_LEVEL else "Not significant"}')

# Correlation: alcohol vs grades
if 'Walc' in df.columns and 'G3' in df.columns:
    r, p = stats.spearmanr(df['Walc'], df['G3'])
    print(f'\nSpearman (Walc vs G3): rho={r:.4f}, p={p:.4f}')

if 'Walc' in df.columns and 'absences' in df.columns:
    r, p = stats.spearmanr(df['Walc'], df['absences'])
    print(f'Spearman (Walc vs absences): rho={r:.4f}, p={p:.4f}')

Mann-Whitney U (Male vs Female Walc): U=24838, p=0.0000
Result: Significant

Spearman (Walc vs G3): rho=-0.1045, p=0.0380
Spearman (Walc vs absences): rho=0.2085, p=0.0000


In [12]:
num_cols = df.select_dtypes(include=[np.number]).columns
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(df[num_cols].corr(), cmap='coolwarm', center=0, fmt='.2f',
            ax=ax, linewidths=0.5, annot=True, annot_kws={'size': 7})
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [13]:
if 'goout' in df.columns and 'Walc' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df, x='goout', y='Walc', ax=axes[0], palette='viridis')
    axes[0].set_title('Weekend Alcohol by Going Out Frequency')
    
    if 'freetime' in df.columns:
        sns.boxplot(data=df, x='freetime', y='Walc', ax=axes[1], palette='Set2')
        axes[1].set_title('Weekend Alcohol by Free Time')
    
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Most students report low alcohol consumption** (1-2 on the 1-5 scale)
2. **Weekend drinking is higher than weekday** — as expected for students
3. **Males tend to consume more alcohol** than females
4. **Going out frequency strongly correlates** with alcohol consumption
5. **Higher alcohol consumption is weakly associated with lower grades**
6. **Absences tend to increase** with higher alcohol consumption

## Limitations

1. **Self-reported data:** Students may underreport alcohol use
2. **Portuguese students only:** Results may not generalize globally
3. **Cross-sectional:** Cannot establish causation
4. **Ordinal scale:** 1-5 scales limit statistical precision
5. **Small sample:** ~395 students

## Recommendations / Next Steps

1. Build a classification model to identify at-risk students
2. Combine math and Portuguese datasets for richer analysis
3. Investigate interaction effects (e.g., does parental education moderate the alcohol-grades relationship?)

### How to Extend This Analysis
- Apply logistic regression to predict high consumption
- Create risk profiles using clustering
- Analyze longitudinal patterns if follow-up data is available

## Common Mistakes

1. **Using Pearson correlation for ordinal data:** Use Spearman rank correlation instead
2. **Treating alcohol levels as continuous:** They're ordinal — use appropriate tests
3. **Claiming causation:** This is observational data — alcohol doesn't 'cause' lower grades
4. **Ignoring confounders:** Family background, school type, and socioeconomic factors all play roles
5. **Not combining Dalc and Walc thoughtfully:** Analyze both separately and together

## Mini Challenge / Exercises

1. Which parental education level is associated with the lowest student alcohol consumption?
2. Is there a significant relationship between romantic relationships and alcohol consumption?
3. Build a logistic regression predicting high_drinker from available features
4. Create a profile of the 'typical high consumer' vs 'typical low consumer'
5. What percentage of grade variance is explained by alcohol consumption alone?

In [14]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Most students report modest alcohol consumption**, but a meaningful minority drinks heavily
- **Social factors** (going out, free time) are stronger predictors than academic factors
- **Gender differences exist** but shouldn't be oversimplified
- **The relationship between alcohol and grades is weak** — many other factors matter more
- **Ordinal data requires special treatment** — use appropriate statistical tests

This analysis demonstrates social science EDA techniques on sensitive survey data.